# PathQ-Former — BLCA 5-Fold CV Training (Colab Free)

**Runtime:** Go to `Runtime > Change runtime type > T4 GPU`

**Resume support:** If the session disconnects, just rerun all cells.
Completed folds are saved to Google Drive and automatically skipped.
Mid-fold progress is also checkpointed every epoch.

In [ ]:
# ============================================================
# Cell 1: Mount Drive (checkpoints + progress persist here)
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR = '/content/drive/MyDrive/PathQFormer/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Check if we have progress from a previous session
progress_file = os.path.join(OUTPUT_DIR, 'blca', 'cv_progress.json')
if os.path.exists(progress_file):
    import json
    with open(progress_file) as f:
        prev = json.load(f)
    done = prev.get('fold_results', {})
    print(f'Previous progress found: {len(done)}/5 folds complete')
    for fold, ci in sorted(done.items()):
        print(f'  Fold {fold}: C-Index = {ci:.4f}')
    if len(done) >= 5:
        print('\nAll folds complete! Skip to Cell 7 for results.')
else:
    print('No previous progress — starting fresh')

In [ ]:
# ============================================================
# Cell 2: Check GPU and disk
# ============================================================
import torch, shutil

assert torch.cuda.is_available(), 'No GPU! Go to Runtime > Change runtime type > T4'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

total, used, free = shutil.disk_usage('/content')
print(f'Disk: {free / 1e9:.1f} GB free / {total / 1e9:.1f} GB total')

In [ ]:
# ============================================================
# Cell 3: Unzip source code + install deps
# ============================================================
os.chdir('/content')

if not os.path.exists('/content/PathQFormer/src'):
    ZIP_PATH = '/content/drive/MyDrive/PathQFormer/PathQFormer_src.zip'
    if os.path.exists(ZIP_PATH):
        !unzip -qo "{ZIP_PATH}" -d /content/
    else:
        raise FileNotFoundError(
            f'{ZIP_PATH} not found.\n'
            'Upload PathQFormer_src.zip to Google Drive at MyDrive/PathQFormer/'
        )

os.chdir('/content/PathQFormer')
!pip install -q lifelines h5py pyyaml scikit-survival
print('Setup complete')

In [ ]:
# ============================================================
# Cell 4: Download BLCA embeddings to Colab local disk
# ~30 GB download + extraction. Takes ~10-15 min.
# Must re-download each new session (local disk is ephemeral).
# ============================================================
import glob

EMBEDDINGS_DIR = '/content/embeddings_blca'
h5_files = glob.glob(f'{EMBEDDINGS_DIR}/**/*.h5', recursive=True)

if len(h5_files) < 400:
    !pip install -q huggingface_hub
    from huggingface_hub import login
    login()  # Paste your HF token when prompted
    
    print('Downloading TCGA-BLCA embeddings (~30 GB)...')
    from huggingface_hub import hf_hub_download
    tar_path = hf_hub_download(
        repo_id='MahmoodLab/UNI2-h-features',
        filename='TCGA/TCGA-BLCA.tar.gz',
        repo_type='dataset',
        local_dir='/content/hf_cache',
    )
    
    print('Extracting...')
    os.makedirs(EMBEDDINGS_DIR, exist_ok=True)
    !tar -xzf "{tar_path}" -C "{EMBEDDINGS_DIR}"
    !rm -rf /content/hf_cache
    
    h5_files = glob.glob(f'{EMBEDDINGS_DIR}/**/*.h5', recursive=True)
    print(f'Done — extracted {len(h5_files)} slides')
else:
    print(f'Embeddings already present ({len(h5_files)} slides)')

# Find the directory containing h5 files
if h5_files:
    EMBEDDINGS_DIR = os.path.dirname(h5_files[0])
print(f'Embeddings dir: {EMBEDDINGS_DIR}')

In [ ]:
# ============================================================
# Cell 5: Verify data
# ============================================================
import h5py, pandas as pd

with h5py.File(h5_files[0], 'r') as f:
    print(f'Sample: {os.path.basename(h5_files[0])}')
    print(f'Shape: {f["features"].shape} | Dtype: {f["features"].dtype}')

meta = pd.read_csv('data/survpath_repo/datasets_csv/metadata/tcga_blca.csv')
print(f'BLCA metadata: {len(meta)} rows')

In [ ]:
# ============================================================
# Cell 6: Run 5-fold CV (auto-resumes from last completed fold)
# ============================================================
from src.training.train import train_cv

cfg = {
    # Data
    'survpath_dir': 'data/survpath_repo',
    'embeddings_dir': EMBEDDINGS_DIR,
    'output_dir': OUTPUT_DIR,
    'cancer_type': 'blca',
    'pathway_type': 'combine',
    
    # Architecture
    'wsi_input_dim': 1536,
    'hidden_dim': 256,
    'num_queries': 32,
    'num_heads': 8,
    'query_layers': 2,
    'fusion_layers': 2,
    'num_bins': 4,
    
    # Regularization
    'dropout': 0.1,
    'modality_dropout': 0.15,
    
    # Training
    'lr': 2e-4,
    'weight_decay': 1e-5,
    'batch_size': 1,
    'epochs': 20,
    'patience': 5,
    'num_folds': 5,
    'num_workers': 2,
    'nll_alpha': 0.0,
    
    # Pathway tokenizer
    'pathway_hidden_dim': 128,
    'min_genes': 3,
    'max_genes': 300,
    
    'max_patches': None,
}

results = train_cv(cfg)

In [ ]:
# ============================================================
# Cell 7: Results summary
# ============================================================
import numpy as np, json

# Load from progress file (works even if Cell 6 was run in a prior session)
progress_file = os.path.join(OUTPUT_DIR, 'blca', 'cv_progress.json')
if os.path.exists(progress_file):
    with open(progress_file) as f:
        progress = json.load(f)
    results = [progress['fold_results'][str(i)] for i in range(5)]

print('\n' + '='*60)
print('  PathQ-Former — BLCA 5-Fold CV Results')
print('='*60)
for i, ci in enumerate(results):
    print(f'  Fold {i}: C-Index = {ci:.4f}')
print(f'\n  Mean: {np.mean(results):.4f} \u00b1 {np.std(results):.4f}')
print('='*60)

# Save final results JSON
results_path = os.path.join(OUTPUT_DIR, 'blca_results.json')
with open(results_path, 'w') as f:
    json.dump({
        'cancer_type': 'BLCA',
        'fold_cindices': results,
        'mean_cindex': float(np.mean(results)),
        'std_cindex': float(np.std(results)),
        'config': cfg,
    }, f, indent=2)
print(f'Saved to {results_path}')